# Imports

In [1]:
import re
import optuna
import pickle

import numpy as np
import pandas as pd

from sklearn import set_config
from catboost import CatBoostClassifier

from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import StackingClassifier, VotingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict

from feature_engine.selection import DropFeatures

/home/junior/Documentos/GitHub/prediction-f1-pit-stops-kaggle-competition/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Utils

In [2]:
set_config(enable_metadata_routing=True)

# Loading Dataset

In [3]:
X_train = pd.read_parquet('../data/processed/X_train_stacking2.parquet')
y_train = pd.read_parquet('../data/processed/y_train.parquet')

X_test = pd.read_parquet('../data/processed/X_test_stacking2.parquet')

In [4]:
X_train.head()

,lgbm_proba,cat_proba,xgb_proba,hist_proba,lg_proba,knn_proba,ridge_proba,voting_lgbm_cat_xgb_hist_proba,voting_lg_ridge_proba,stacking_lgbm_with_knn_and_voting_lg_ridge_and_voting_lgbm_cat_xgb_hist_proba
0,0.802607,0.891870,0.895659,0.883947,0.808497,0.887454,0.519677,0.868510,0.664963,0.772948
1,0.649405,0.831360,0.829854,0.779541,0.689888,0.519966,0.395214,0.772514,0.543445,0.595178
2,0.505904,0.851809,0.751522,0.758911,0.558520,0.209855,0.259447,0.717012,0.409891,0.515461
3,0.001713,0.005910,0.007057,0.007298,0.002275,0.000000,0.005106,0.005494,0.003682,0.001281
4,0.392205,0.565506,0.489033,0.482967,0.414144,0.297043,0.150282,0.482416,0.283014,0.293768


In [5]:
X_test.head()

,lgbm_proba,cat_proba,xgb_proba,hist_proba,lg_proba,knn_proba,ridge_proba,voting_lgbm_cat_xgb_hist_proba,voting_lg_ridge_proba,stacking_lgbm_with_knn_and_voting_lg_ridge_and_voting_lgbm_cat_xgb_hist_proba
0,0.004591,0.031986,0.022774,0.020057,0.006419,0.000000,0.004726,0.019849,0.005577,0.006190
1,0.003032,0.011817,0.014957,0.018887,0.063872,0.117213,0.033245,0.012173,0.048652,0.003070
2,0.003777,0.011641,0.014307,0.011497,0.004517,0.000000,0.003903,0.010304,0.004212,0.003695
3,0.163131,0.544040,0.585898,0.485388,0.655870,0.099916,0.296579,0.444558,0.477315,0.232534
4,0.872172,0.967646,0.959451,0.959356,0.840450,1.000000,0.548189,0.939648,0.695206,0.894775


# Machine Learning

In [6]:
def objective(trial, X, y):

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    aucs = []

    params = {
        "solver": trial.suggest_categorical("solver", ["saga"]),
        "C": trial.suggest_float("C", 1e-5, 100, log=True),
        "l1_ratio": trial.suggest_float("l1_ratio", 0.0, 1.0),
        "class_weight": trial.suggest_categorical("class_weight", [None, "balanced"]),
        "fit_intercept": trial.suggest_categorical("fit_intercept", [True, False]),
        "tol": trial.suggest_float("tol", 1e-6, 1e-2, log=True),
        "max_iter": trial.suggest_int("max_iter", 1000, 5000)
    }

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y)):

        X_train, X_valid = X.iloc[train_idx, :], X.iloc[valid_idx, :]
        y_train, y_valid = y.iloc[train_idx, 0], y.iloc[valid_idx, 0]

        model = LogisticRegression(**params)
        model.fit(X_train, y_train)

        proba = model.predict_proba(X_valid)[:, 1]

        auc = roc_auc_score(y_valid, proba)
        aucs.append(auc)

        trial.report(np.mean(aucs), step=fold)

        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return np.mean(aucs)

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42), pruner=optuna.pruners.MedianPruner(n_warmup_steps=2))
study.optimize(lambda trial: objective(trial, X_train, y_train), n_trials=500, n_jobs=-1, show_progress_bar=True)


print("Best trial score:")
print(study.best_trial.value)

print("\nBest params:")
print(study.best_trial.params)

[I 2026-05-19 11:37:32,165] A new study created in memory with name: no-name-848536ac-2236-4bbd-94a8-b1aa1993c2bf
Best trial: 10. Best value: 0.5:   0%|▏                                                                                                                     | 1/500 [00:06<57:15,  6.88s/it]

[I 2026-05-19 11:37:39,049] Trial 10 finished with value: 0.5 and parameters: {'solver': 'saga', 'C': 1.2905632598245504e-05, 'l1_ratio': 0.8220212359286581, 'class_weight': 'balanced', 'fit_intercept': True, 'tol': 1.6115877313136515e-06, 'max_iter': 1273}. Best is trial 10 with value: 0.5.


Best trial: 8. Best value: 0.805073:   0%|▍                                                                                                                 | 2/500 [00:09<36:25,  4.39s/it]

[I 2026-05-19 11:37:41,679] Trial 8 finished with value: 0.8050733585051886 and parameters: {'solver': 'saga', 'C': 0.0002033376587658991, 'l1_ratio': 0.9496843244387186, 'class_weight': None, 'fit_intercept': False, 'tol': 0.004601133976411335, 'max_iter': 1773}. Best is trial 8 with value: 0.8050733585051886.


Best trial: 5. Best value: 0.910234:   1%|▋                                                                                                                 | 3/500 [00:10<21:50,  2.64s/it]

...

[I 2026-05-19 11:53:28,438] Trial 498 finished with value: 0.9497127015867639 and parameters: {'solver': 'saga', 'C': 0.0031551653601360656, 'l1_ratio': 0.9114924197926617, 'class_weight': None, 'fit_intercept': True, 'tol': 0.00011112079250789059, 'max_iter': 1821}. Best is trial 93 with value: 0.94971693243582.


Best trial: 93. Best value: 0.949717: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████▊| 499/500 [15:57<00:01,  1.20s/it]

[I 2026-05-19 11:53:29,186] Trial 496 finished with value: 0.9497127811322603 and parameters: {'solver': 'saga', 'C': 0.0031325468395663613, 'l1_ratio': 0.9110802428832693, 'class_weight': None, 'fit_intercept': True, 'tol': 3.5517986947942004e-05, 'max_iter': 1596}. Best is trial 93 with value: 0.94971693243582.


Best trial: 93. Best value: 0.949717: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████| 500/500 [15:58<00:00,  1.92s/it]

[I 2026-05-19 11:53:30,380] Trial 499 finished with value: 0.9497131510453405 and parameters: {'solver': 'saga', 'C': 0.0030465541891385064, 'l1_ratio': 0.9139785509303224, 'class_weight': None, 'fit_intercept': True, 'tol': 3.598636390590139e-05, 'max_iter': 4935}. Best is trial 93 with value: 0.94971693243582.
Best trial score:
0.94971693243582

Best params:
{'solver': 'saga', 'C': 0.002193203638990036, 'l1_ratio': 0.89839712471031, 'class_weight': None, 'fit_intercept': True, 'tol': 0.0001350258081959178, 'max_iter': 4918}


In [7]:
stacking_lg = LogisticRegression(**study.best_params).fit(X_train, y_train.PitNextLap)

# Submission

In [8]:
submission = pd.read_csv('../data/raw/sample_submission.csv')

In [9]:
submission['PitNextLap'] = stacking_lg.predict_proba(X_test)[:, 1]

In [10]:
submission.to_csv('../data/submission/submission.csv', index=False)

In [11]:
X_train.columns.tolist()

['lgbm_proba',
 'cat_proba',
 'xgb_proba',
 'hist_proba',
 'lg_proba',
 'knn_proba',
 'ridge_proba',
 'voting_lgbm_cat_xgb_hist_proba',
 'voting_lg_ridge_proba',
 'stacking_lgbm_with_knn_and_voting_lg_ridge_and_voting_lgbm_cat_xgb_hist_proba']